In [1]:
!pip install streamlit -q
!pip install pyngrok -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 32.0 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import Pipeline
import joblib

# [조건 1] 데이터 준비 및 소규모 훈련 샘플링
# 데이터를 로드하고 결측치를 제거한다.
df = pd.read_csv(r"https://github.com/dongupak/DataML/raw/main/csv/life_expectancy.csv")
df.dropna(inplace=True) # 결측치를 제거하고 제거한 결과를 inplace  = True로 다시 df에 저장

# 기대수명을 예측하기 위한 독립변수 3개 선택
X = df[['BMI','GDP','Alcohol']].values
y = df['Life expectancy'].values

# 전체 데이터를 80% 훈련(train), 20% 테스트(test) 세트로 분할
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 훈련 데이터는 무작위로 50개 샘플만 추출하여 학습에 사용
np.random.seed(42)
sample_idx = np.random.choice(len(X_train), size=50, replace=False) # 무작위 50개 샘플 추출
X_train_sample = X_train[sample_idx]
y_train_sample = y_train[sample_idx]

# [조건 2] 파이프라인 기반 모델 3종 학습 및 저장 (1단계)
models = [
    ("Linear", Pipeline([
        ("poly", PolynomialFeatures(degree=1)),
        ("scaler", StandardScaler()),
        ("regr", LinearRegression())
    ])), # Model 1 (Linear): 1차 항 기본 선형 회귀 파이프라인
    ("Poly", Pipeline([
        ("poly", PolynomialFeatures(degree=3)),
        ("scaler", StandardScaler()),
        ("regr", LinearRegression())
    ])), # Model 2 (Poly): 3차 다항 회귀 파이프라인 (규제 없음 → 과대적합 유도)
    ("Ridge", Pipeline([
        ("poly", PolynomialFeatures(degree=3)),
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=1)) # Model 3 (Ridge): 3차 다항 회귀 + 릿지 규제 파이프라인 (alpha=1.0)
    ]))
]

# 결정계수와 MSE 계산과 특성갯수 계산
results = []

for model_name, model in models:
    model.fit(X_train_sample, y_train_sample) # 모델 학습시키기

    y_pred_train = model.predict(X_train_sample) # 학습 데이터의 예상치
    y_pred_test = model.predict(X_test) # 테스트 데이터의 예상치

    r2_train = r2_score(y_train_sample, y_pred_train) # 학습데이터 R2 결정계수
    r2_test = r2_score(y_test, y_pred_test)   # 테스트 데이터 R2 결정계수

    mse_train = mean_squared_error(y_train_sample, y_pred_train) # 학습테이터 결측치
    mse_test = mean_squared_error(y_test, y_pred_test)   # 테스트 데이터 결측치
    poly_step = model.named_steps['poly']
    complexity = poly_step.n_output_features_  #몇 개의 특성(feature)이 만들어졌는지 계산

    results.append([
        model_name,
        r2_train,
        r2_test,
        mse_train,
        mse_test,
        complexity
    ])
    joblib.dump(model, f"{model_name}.pkl")

result_df = pd.DataFrame(
    results,
    columns=[
        "Model",
        "Train R2",
        "Test R2",
        "Train MSE",
        "Test MSE",
        "Complexity"
    ]
)

print(result_df)
joblib.dump(result_df, "results.pkl")

    Model  Train R2    Test R2  Train MSE     Test MSE  Complexity
0  Linear  0.484473   0.267066  56.535069    52.054788           4
1    Poly  0.693214 -33.421405  33.643647  2444.693696          20
2   Ridge  0.554887   0.325826  48.813171    47.881497          20


['results.pkl']

In [21]:
%%writefile app.py

import streamlit as st
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

results = joblib.load("results.pkl")

models = {
    "Linear": joblib.load("Linear.pkl"),
    "Poly": joblib.load("Poly.pkl"),
    "Ridge": joblib.load("Ridge.pkl")
}
st.header("모델 성능 비교")
st.dataframe(results) # R2, MSE, 특성개수 표시

# 막대그래프 시각화
fig, ax = plt.subplots()
results.plot(kind='bar', x='Model', y='Test R2', ax=ax, color='skyblue')
ax.set_title("Model Test R^2 Comparison")
st.pyplot(fig)

# 3. [조건 4] 실시간 예측 UI 구성 (사이드바)
st.sidebar.header("입력 특성 조절")
bmi = st.sidebar.slider("BMI", 10.0, 50.0, 25.0)
gdp = st.sidebar.slider("GDP", 0.0, 100000.0, 5000.0)
alcohol = st.sidebar.slider("Alcohol", 0.0, 20.0, 5.0)

# 선택된 특성을 데이터프레임으로 변환
input_data = pd.DataFrame([[bmi, gdp, alcohol]], columns=['BMI', 'GDP', 'Alcohol'])

# 모델 선택
selected_model_name = st.selectbox("사용할 모델 선택", ["Linear", "Poly", "Ridge"])
selected_model = models[selected_model_name]

# 예측
if st.button("예측하기"):
    prediction = selected_model.predict(input_data)
    st.subheader(f"예측된 기대수명: {prediction[0]:.2f} 세")

Overwriting app.py


In [22]:
# 기존 코드를 멈추고, 아래 코드로 다시 실행.
!pip install streamlit pyngrok -q
from pyngrok import ngrok
import os

# 1. 혹시 열려있을지 모르는 기존 터널 모두 닫기

ngrok.kill()
# 2. 인증 토큰 설정 (여기에 본인의 토큰을 붙이기)
ngrok.set_auth_token("3Ewe8JYrwIumAoBprPJIOsJe4s8_3mDBsSw983hyyCeCTjVus")

# 3. Streamlit 실행 (서버 주소를 명시적으로 127.0.0.1로 고정)
os.system("streamlit run app.py --server.address 127.0.0.1 &")

# 4. ngrok 터널 연결
public_url = ngrok.connect(8501) #(8501 포트의 127.0.0.1 주소를 바라보게 함)
print(f"아래 링크를 클릭하세요:\n{public_url}")

아래 링크를 클릭하세요:
NgrokTunnel: "https://example-pelican-baggage.ngrok-free.dev" -> "http://localhost:8501"
